# T-ChIC modality prediction benchmark

Run models that predict RNA expression from T-ChIC histone measurements and compare their official OpenProblems metric scores.

## Workflow
1. Select the dataset and saved plate-aware split.
2. Prepare or reuse the benchmark input files.
3. Select models and run settings.
4. Run the OpenProblems benchmark.
5. Save and compare the official results.

For development runs, the benchmark's `test_mod1` and `test_mod2` files contain our validation cells. The final-test partition remains reserved for the final evaluation.

The workflow is first build using the existing prepared K4 dataset and the mean-per-gene baseline.

## 1. Data preparation

Choose the repository, dataset and related split file

In [ ]:
from pathlib import Path

REPO = Path(
    "/home/michiel/thesis/benchmark/task_predict_modality_upstream_validation"
)
SOURCE = Path(
    "/home/vivek/NOBINFBACKUP/projects/mGast_tchic/adata_files"
)

DATASET = "h3k4me3_genelevel_dev"

DATA_SETTINGS = {
    "raw_path": SOURCE / "allsamples_genelevel_RNAandChIC_counts.h5ad",
    "filtered_path": SOURCE / "allsamples_genelevel_RNAandChIC_counts.filt-norm.h5ad",
    "histone_mark": "H3K4me3",
    "histone_layer": "chic_counts",
    "rna_layer": "unspliced",
    "normalization": "log_cp10k",
    "evaluation_split": "validation",
}

SPLIT_SETTINGS = {
    "path": REPO / "outputs/tchic/splits/h3k4me3_plate_split.csv",
    "strategy": "plate_aware",
    "seed": 42,
    "search_iterations": 150_000,

    # Allocation order: train, validation, test.
    "target_proportions": [0.70, 0.15, 0.15],
    "target_plate_counts": [42, 9, 9],

    "expected_plates_per_day": {
        0: 6, 3: 6, 4: 12, 5: 12, 6: 12, 7: 12,
    },

    # Each allocation is assigned to one day with that plate count.
    "day_allocations": {
        6: [(4, 1, 1), (4, 1, 1)],
        12: [(8, 2, 2), (8, 2, 2), (9, 1, 2), (9, 2, 1)],
    },

    "required_coverage": ["rep", "seqrun"],
    "balance_weights": {
        "within_day_cell_rmse": 1.0,
        "overall_cell_rmse": 1.0,
        "day_rep_rmse": 0.25,
        "rep_rmse": 0.10,
        "seqrun_rmse": 0.10,
    },
}

PREPARED = REPO / "outputs/tchic/prepared" / DATASET
SPLIT = SPLIT_SETTINGS["path"]

print(f"Dataset: {DATASET}")
print(f"Task: {DATA_SETTINGS['histone_mark']} → {DATA_SETTINGS['rna_layer']} RNA")
print(f"Evaluation partition: {DATA_SETTINGS['evaluation_split']}")

## Prepare or reuse benchmark data

Use the selected source data, split and normalization to prepare the
benchmark inputs. Reuse existing files only when their preparation
settings and validation records match.

In [ ]:
import sys
import pandas as pd

scripts_dir = str(REPO / "scripts")
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

import tchic_benchmark

print(f"Preparation code: {tchic_benchmark.__file__}")

In [ ]:
import importlib

importlib.reload(tchic_benchmark)

PREPARED = tchic_benchmark.prepare_benchmark_data(
    data_settings=DATA_SETTINGS,
    split_path=SPLIT,
    prepared_dir=PREPARED,
    dataset_id=f"tchic_mgast_{DATASET}",
)

## 2. Select a model and name the run

METHOD = the component name recognized by the OpenProblems benchmark.

RUN_NAME = the name of the run to your liking.

This code only selects the settings, it does not yet make any files and does not yet start a run.

In [ ]:
METHOD = "mean_per_gene"
RUN_NAME = "mean_per_gene_002"

RUN_DIR = REPO / "outputs/tchic/runs" / DATASET / RUN_NAME

print(f"Model: {METHOD}")
print(f"Run: {RUN_NAME}")
print(f"Output directory: {RUN_DIR}")

## 3. Select the benchmark workflow and runtime configuration

Reuse the built OpenProblems workflow and the runtime configuration
from notebook 3. This configuration selects the containers and Python
environment used to execute the baseline and metric components.

In [ ]:
WORKFLOW = REPO / "target_benchmark_complete/nextflow/workflows/run_benchmark"
RUNTIME_CONFIG = REPO / "outputs/tchic/runtime/mean_per_gene.config"

for path in (WORKFLOW / "main.nf", RUNTIME_CONFIG):
    if not path.is_file():
        raise FileNotFoundError(f"Required file not found: {path}")

print(f"Workflow: {WORKFLOW}")
print(f"\nRuntime configuration:\n{RUNTIME_CONFIG.read_text()}")

## 4. Prepare the benchmark run

Create a separate run directory and save the runtime configuration
and executable command. The workflow will run the selected model,
calculate the official metrics and collect the results.

In [ ]:
import shlex
import shutil

# Require a new directory to protect previous runs.
if RUN_DIR.exists():
    raise FileExistsError(
        f"Run directory already exists: {RUN_DIR}\n"
        "Choose a new RUN_NAME for a new experiment."
    )

RUN_DIR.mkdir(parents=True)
RUN_CONFIG = RUN_DIR / "runtime.config"
shutil.copy2(RUNTIME_CONFIG, RUN_CONFIG)
shutil.copy2(SPLIT, RUN_DIR / "plate_split.csv")

command = [
    "nextflow", "run", str(WORKFLOW),
    "-profile", "singularity",
    "-c", str(RUN_CONFIG),
    "-work-dir", str(RUN_DIR / "work"),
    "-with-trace", str(RUN_DIR / "trace.txt"),
    "--method_ids", METHOD,
    "--publish_dir", str(RUN_DIR / "publish"),
]

for part in ("train", "test"):
    for mod in ("mod1", "mod2"):
        command += [
            f"--input_{part}_{mod}",
            str(PREPARED / f"{part}_{mod}.h5ad"),
        ]

outputs = {
    "scores": "score_uns.yaml",
    "method_configs": "method_configs.yaml",
    "metric_configs": "metric_configs.yaml",
    "dataset_info": "dataset_uns.yaml",
    "task_info": "task_info.yaml",
}
for name, filename in outputs.items():
    command += [f"--output_{name}", filename]

script = RUN_DIR / "run_command.sh"
script.write_text(
    "#!/usr/bin/env bash\n"
    "set -euo pipefail\n"
    f"cd {shlex.quote(str(RUN_DIR))}\n"
    "export NXF_SYNTAX_PARSER=v1\n"
    + shlex.join(command) + "\n"
)
script.chmod(0o755)

print(f"Run prepared: {RUN_NAME}")
print(f"Saved command: {script}")
print("Ready to run; no model has been executed yet.")

## 5. Run the benchmark

Execute the saved command: run the model, calculate the official
metrics and collect the results. Save the execution log alongside
the run configuration.

In [ ]:
import os
import subprocess

log_path = RUN_DIR / "run.log"
env = os.environ.copy()
env["NXF_ANSI_LOG"] = "false"

with log_path.open("x") as log:
    with subprocess.Popen(
        ["bash", str(RUN_DIR / "run_command.sh")],
        cwd=RUN_DIR,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()

        returncode = process.wait()

if returncode != 0:
    raise RuntimeError(
        f"Benchmark failed with exit code {returncode}. See {log_path}"
    )

score_path = RUN_DIR / "publish/score_uns.yaml"
if not score_path.is_file():
    raise FileNotFoundError(f"Expected score file not found: {score_path}")

print(f"\nBenchmark completed: {RUN_NAME}")
print(f"Official scores: {score_path}")

## 6. Save and display official scores

Read the scores produced by OpenProblems without recalculating them.
Save one row per metric in a CSV file and display a compact comparison
table with one row per run.

In [ ]:
import pandas as pd
import yaml

score_path = RUN_DIR / "publish/score_uns.yaml"
records = yaml.safe_load(score_path.read_text())

rows = []
for record in records:
    if len(record["metric_ids"]) != len(record["metric_values"]):
        raise ValueError("Metric names and values have different lengths.")

    for metric, value in zip(record["metric_ids"], record["metric_values"]):
        rows.append({
            "run": RUN_NAME,
            "dataset_id": record["dataset_id"],
            "method_id": record["method_id"],
            "metric": metric,
            "value": value,
            "score_source": str(score_path),
        })

results = pd.DataFrame(rows)
summary = results.pivot(
    index=["run", "dataset_id", "method_id"],
    columns="metric",
    values="value",
)

results.to_csv(RUN_DIR / "metrics.csv", index=False)
print(f"Run: {RUN_NAME}")
display(
    results[["metric", "value"]]
    .set_index("metric")
    .rename(columns={"value": "Score"})
    .round(6)
)
print(f"Saved results: {RUN_DIR / 'metrics.csv'}")

## 7. Plot official metric scores

Show each metric separately because their scales and interpretations
differ. Higher correlations are better; lower RMSE and MAE are better.
These are final validation scores, not training curves.

In [ ]:
import matplotlib.pyplot as plt

plot_data = results.pivot(
    index="run", columns="metric", values="value"
)

fig, axes = plt.subplots(4, 2, figsize=(7, 10), layout="constrained")

for ax, metric in zip(axes.flat, plot_data.columns):
    values = plot_data[metric]
    bars = ax.bar(values.index, values, width=0.45, color="#0072B2")

    direction = "↓ lower is better" if metric in ("rmse", "mae") else "↑ higher is better"
    ax.set_title(f"{metric.replace('_', ' ')}\n{direction}", fontsize=10)
    ax.tick_params(axis="both", labelsize=8)
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.2)
    ax.set_xlim(-0.7, len(values) - 0.3)

    if metric not in ("rmse", "mae"):
        ax.set_ylim(-1 if (values < 0).any() else 0, 1)
    else:
        ax.set_ylim(0, max(values.max() * 1.2, 0.01))

figure_path = RUN_DIR / "metric_scores.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()